In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_community.agent_toolkits import create_sql_agent


In [ ]:
# =====================================================================
# DATABASE CONNECTION (CRITICAL SECURITY STEP: USE A READ-ONLY USER)
# =====================================================================

db_user = "root"
db_password = os.getenv("DB_PASSWORD")  # Replace with your actual password or use environment variables for better security
db_host = "localhost"
db_name = "atliq_tshirts"

db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}",sample_rows_in_table_info=3)

print(db.table_info)

In [ ]:
load_dotenv()  # Load environment variables from .env file

# 1. Create a bulletproof system prompt detailing your database quirks
sql_system_prompt = (
    "You are a MySQL data analyst agent. Review the user's query and generate "
    "a highly accurate MySQL statement based on these strict database rules:\n\n"
    
    "===== CRITICAL DATA MAPPING RULES =====\n"
    "1. SIZE COLUMN: The `size` column ONLY stores short codes. You MUST map full-word "
    "sizes to these exact uppercase strings:\n"
    "   - 'extra small' or 'xs' -> 'XS'\n"
    "   - 'small' or 's'       -> 'S'\n"
    "   - 'medium' or 'm'      -> 'M'\n"
    "   - 'large' or 'l'       -> 'L'\n"
    "   - 'extra large' or 'xl'-> 'XL'\n\n"
    
    "2. CASE INSENSITIVITY (BRAND & COLOR):\n"
    "   To prevent mismatches with capitalization (e.g., 'Nike' vs 'nike'), always wrap "
    "the column name in the MySQL `LOWER()` function and compare it against a lowercase string.\n"
    "   Example: `LOWER(brand) = 'nike'` and `LOWER(color) = 'white'`\n\n"
    
    "===== FEW-SHOT EXAMPLE =====\n"
    "User Question: How many t-shirts do we have left for nike in extra small size and white color?\n"
    "Correct SQL Query: SELECT COUNT(*) FROM products WHERE LOWER(brand) = 'nike' AND size = 'XS' AND LOWER(color) = 'white';\n\n"
    
    "Now, process the user's input question using the rules above. Only output read-only SELECT queries."
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile", # High-reasoning 70B model
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0 # CRITICAL: keep at 0 for deterministic SQL generation
)

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()



In [ ]:
agent_executor = create_sql_agent(llm, toolkit=toolkit, agent_type="tool-calling", state_modifier=sql_system_prompt, verbose=True)

safe_prompt = "How many t-shirts do we have left for nike in extra small size and white color?"
response = agent_executor.invoke({"input": safe_prompt})
print(response)

In [15]:
qns2 = "How much is the price of the inventory for all small size t-shirts?"
response2 = agent_executor.invoke({"input": qns2})
print(response2)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`
responded:   Then I should double check my query before running it to get the answer.



discounts, t_shirts
Invoking: `sql_db_schema` with `{'table_names': 'table1, table2, table3'}`
responded:   Then I should double check my query before running it to get the answer.



Error: table_names {'table3', 'table2', 'table1'} not found in database
Invoking: `sql_db_query_checker` with `{'query': "SELECT price FROM table_name WHERE size = 'small' AND type = 't-shirt'"}`
responded:   Then I should double check my query before running it to get the answer.



```sql
SELECT price 
FROM table_name 
WHERE size = 'small' 
AND type = 't-shirt'
```
Invoking: `sql_db_query` with `{'query': "SELECT price FROM table_name WHERE size = 'small' AND type = 't-shirt'"}`
responded:   Then I should double check my query before running it to get the answer.



Error: (pymysql.err.ProgrammingError) (1146, "Tab